In [1]:
import numpy as np
import scipy.sparse as sp
from scipy.special import genlaguerre, gammaln
import matplotlib.pyplot as plt
import scienceplots
plt.style.use(['science', 'notebook'])
from scipy.sparse.linalg import eigsh

In [2]:
# ---------------------------------------------------------------------------
# Primitive sparse local operators
# ---------------------------------------------------------------------------

def _eye(d):
    return sp.eye(d, format='csr')

def _destroy(d):
    data = np.sqrt(np.arange(1, d, dtype=float))
    return sp.diags(data, 1, shape=(d, d), format='csr')

def _num(d):
    return sp.diags(np.arange(d, dtype=float), 0, format='csr')


# ---------------------------------------------------------------------------
# Sparse tensor-product embedding
# ---------------------------------------------------------------------------

def sparse_embed(ops_at_indices: dict, n_total: int, fock_dim: int) -> sp.csr_matrix:
    """
    Same semantics as your QuTiP embed(), but returns a scipy sparse matrix.
    ops_at_indices: {site_index: sparse_matrix}
    """
    local_ops = [_eye(fock_dim)] * n_total
    for idx, op in ops_at_indices.items():
        local_ops[idx] = op

    result = local_ops[0]
    for op in local_ops[1:]:
        result = sp.kron(result, op, format='csr')
    return result


# ---------------------------------------------------------------------------
# compute_position_exponential  (same logic, returns sparse)
# ---------------------------------------------------------------------------

def compute_position_exponential_sparse(N, u):
    matrix = np.zeros((N, N), dtype=complex)
    for m in range(N):
        for n in range(N):
            if m >= n:
                matrix[m, n] = (
                    np.exp(-u**2 / 2)
                    * np.exp(0.5 * (gammaln(n + 1) - gammaln(m + 1)))
                    * (1.0j * u) ** (m - n)
                    * genlaguerre(n, m - n)(u**2)
                )
            else:
                matrix[m, n] = (
                    np.exp(-u**2 / 2)
                    * np.exp(0.5 * (gammaln(m + 1) - gammaln(n + 1)))
                    * (1.0j * u) ** (n - m)
                    * genlaguerre(m, n - m)(u**2)
                )
    return sp.csr_matrix(matrix)

In [3]:
# ---------------------------------------------------------------------------
# capacitive_ssh  (sparse)
# ---------------------------------------------------------------------------

def capacitive_ssh_sparse(N, fock_transmon, E_c, E_J, Ec1, Ec2):
    capacitive_factor = np.sqrt(E_J * E_c**3) ** 0.25 / np.sqrt(2)
    gc1 = capacitive_factor / Ec1
    gc2 = capacitive_factor / Ec2
    omega_q = np.sqrt(8 * E_c * E_J) - E_c
    Lambda_0 = E_c

    k_values = np.linspace(-np.pi, np.pi, N, endpoint=False)
    delta_ck = np.array([gc1 + np.exp(-1.0j * k) * gc2 for k in k_values])

    delta_ckA = Lambda_0 - np.abs(delta_ck)
    delta_ckB = Lambda_0 + np.abs(delta_ck)
    epsilon_ckA = omega_q + np.abs(delta_ck)
    epsilon_ckB = omega_q - np.abs(delta_ck)

    E_a = np.sqrt(epsilon_ckA**2 - delta_ckA**2)
    E_b = np.sqrt(epsilon_ckB**2 - delta_ckB**2)

    ground_state_energy = 0.5 * np.sum(E_a + E_b) - N * omega_q

    n_op = _num(fock_transmon)
    n_sites = 2 * N
    dim = fock_transmon ** n_sites

    H = sp.csr_matrix((dim, dim), dtype=complex)

    for j in range(N):
        # A-sublattice site: index 2j
        H += E_a[j] * sparse_embed({2 * j: n_op}, n_sites, fock_transmon)
        # B-sublattice site: index 2j+1
        H += E_b[j] * sparse_embed({2 * j + 1: n_op}, n_sites, fock_transmon)

    # ground state energy: scalar * identity
    H += ground_state_energy * sp.eye(dim, format='csr')
    return H


# ---------------------------------------------------------------------------
# finally_transformed_squid_ssh  (sparse)
# ---------------------------------------------------------------------------

def finally_transformed_squid_ssh_sparse(N, fock_transmon, E_c, E_J, Ec1, Ec2, EJ1, EJ2):
    omega_q = np.sqrt(8 * E_c * E_J) - E_c
    squid_factor = 2 * np.sqrt(2 * E_c / E_J)
    gj1 = squid_factor * EJ1
    gj2 = squid_factor * EJ2

    k_values = np.linspace(-np.pi, np.pi, N, endpoint=False)
    capacitive_factor = np.sqrt(E_J * E_c**3) ** 0.25 / np.sqrt(2)
    gc1 = capacitive_factor / Ec1
    gc2 = capacitive_factor / Ec2

    delta_ck = np.array([gc1 + np.exp(-1.0j * k) * gc2 for k in k_values])
    phase_correction = np.angle(delta_ck)
    delta_k = (
        np.array([gj1 + np.exp(-1.0j * k) * gj2 for k in k_values])
        * np.exp(-1.0j * phase_correction)
    )

    diag_coeff_A_0 = (gj1 + gj2) - np.real(delta_k)
    diag_coeff_B_0 = (gj1 + gj2) + np.real(delta_k)
    self_pairing_coeff_A_0 = -0.5 * diag_coeff_B_0
    self_pairing_coeff_B_0 = -0.5 * diag_coeff_A_0
    hopping_coeff_0 = 1.0j * np.imag(delta_k)

    Lambda_0 = E_c
    delta_ckA = Lambda_0 - np.abs(delta_ck)
    delta_ckB = Lambda_0 + np.abs(delta_ck)
    epsilon_ckA = omega_q + np.abs(delta_ck)
    epsilon_ckB = omega_q - np.abs(delta_ck)

    E_a = np.sqrt(epsilon_ckA**2 - delta_ckA**2)
    E_b = np.sqrt(epsilon_ckB**2 - delta_ckB**2)

    cosh_2ra = epsilon_ckA / E_a
    cosh_2rb = epsilon_ckB / E_b
    sinh_2ra = delta_ckA / E_a
    sinh_2rb = delta_ckB / E_b

    ra = 0.5 * np.asinh(sinh_2ra)
    rb = 0.5 * np.asinh(sinh_2rb)

    diag_coeff_A = diag_coeff_A_0 * cosh_2ra + 2.0 * self_pairing_coeff_A_0 * sinh_2ra
    diag_coeff_B = diag_coeff_B_0 * cosh_2rb + 2.0 * self_pairing_coeff_B_0 * sinh_2rb

    self_pairing_coeff_A = self_pairing_coeff_A_0 * cosh_2ra + 0.5 * diag_coeff_A_0 * sinh_2ra
    self_pairing_coeff_B = self_pairing_coeff_B_0 * cosh_2rb + 0.5 * diag_coeff_B_0 * sinh_2rb

    hopping_coeff = hopping_coeff_0 * np.exp(ra + rb)
    # intersite_pairing_coeff == hopping_coeff by construction

    ground_state_energy = (
        0.5 * np.sum(diag_coeff_A + diag_coeff_B - diag_coeff_A_0 - diag_coeff_B_0)
        + N * (0.5 * (gj1 + gj2) - 2.0 * (EJ1 + EJ2))
    )

    b_op = _destroy(fock_transmon)
    bd_op = b_op.T.conj()          # for csr, .conj().T
    n_op = _num(fock_transmon)
    n_sites = 2 * N
    dim = fock_transmon ** n_sites

    H = sp.csr_matrix((dim, dim), dtype=complex)

    for j in range(N):
        mj = (N - j) % N

        # --- diagonal ---
        H += diag_coeff_A[j] * sparse_embed({2 * j:     n_op}, n_sites, fock_transmon)
        H += diag_coeff_B[j] * sparse_embed({2 * j + 1: n_op}, n_sites, fock_transmon)

        # --- self-pairing A  (b†_j b†_{mj}  +  h.c.) ---
        if mj == j:
            op_A = sparse_embed({2 * j: bd_op @ bd_op}, n_sites, fock_transmon)
        else:
            op_A = sparse_embed({2 * j: bd_op, 2 * mj: bd_op}, n_sites, fock_transmon)
        H += self_pairing_coeff_A[j] * op_A
        H += np.conj(self_pairing_coeff_A[j]) * op_A.conj().T

        # --- self-pairing B ---
        if mj == j:
            op_B = sparse_embed({2 * j + 1: bd_op @ bd_op}, n_sites, fock_transmon)
        else:
            op_B = sparse_embed({2 * j + 1: bd_op, 2 * mj + 1: bd_op}, n_sites, fock_transmon)
        H += self_pairing_coeff_B[j] * op_B
        H += np.conj(self_pairing_coeff_B[j]) * op_B.conj().T

        # --- hopping  b†_{A,j} b_{B,j}  +  h.c. ---
        op_hop = sparse_embed({2 * j: bd_op, 2 * j + 1: b_op}, n_sites, fock_transmon)
        H += hopping_coeff[j] * op_hop
        H += np.conj(hopping_coeff[j]) * op_hop.conj().T

        # --- intersite pairing  b†_{A,j} b†_{B,mj}  +  h.c. ---
        op_isp = sparse_embed({2 * j: bd_op, 2 * mj + 1: bd_op}, n_sites, fock_transmon)
        H += hopping_coeff[j] * op_isp
        H += np.conj(hopping_coeff[j]) * op_isp.conj().T

    H += ground_state_energy * sp.eye(dim, format='csr')
    return H

In [4]:
# ---------------------------------------------------------------------------
# ssh_squid_chain  (sparse)
# ---------------------------------------------------------------------------

def ssh_squid_chain_sparse(N, fock_photon, fock_transmon, E_c, E_J, omega_c, gamma, Ec1, Ec2, E_J1, E_J2):
    """
    Sparse version of ssh_squid_chain.

    Hilbert space ordering:  photon ⊗ (transmon_0 ⊗ transmon_1 ⊗ ... ⊗ transmon_{2N-1})
    Total dimension: fock_photon * fock_transmon^(2N)
    """
    dim_transmon = fock_transmon ** (2 * N)
    dim_photon   = fock_photon

    # --- photon sector ---
    H_ph   = omega_c * _num(fock_photon)                          # fock_photon × fock_photon
    sin_op = -0.5j * (
        compute_position_exponential_sparse(fock_photon,  gamma)
        - compute_position_exponential_sparse(fock_photon, -gamma)
    )                                                              # fock_photon × fock_photon

    eye_ph  = _eye(fock_photon)
    eye_tr  = sp.eye(dim_transmon, format='csr')

    # --- chain sectors ---
    H0 = capacitive_ssh_sparse(N, fock_transmon, E_c, E_J, Ec1, Ec2)
    H1 = finally_transformed_squid_ssh_sparse(N, fock_transmon, E_c, E_J, Ec1, Ec2, E_J1, E_J2)

    # --- combine with kron (photon is the leftmost/slowest index) ---
    H = (
        sp.kron(H_ph,   eye_tr,  format='csr')   # H_ph  ⊗ I_transmon
        + sp.kron(eye_ph, H0,    format='csr')   # I_ph  ⊗ H0
        - sp.kron(sin_op, H1,    format='csr')   # sin   ⊗ H1
    )
    return H

In [5]:
# ---------------------------------------------------------------------------
# Quick sanity check against QuTiP  (run as script)
# ---------------------------------------------------------------------------

if __name__ == '__main__':
    import qutip as qt
    import sys

    # Small system so QuTiP is still tractable
    N            = 10
    fock_photon  = 4
    fock_transmon = 2
    E_c  = 0.3
    E_J  = 5.0
    omega_c = 6.0
    gamma   = 0.1
    Ec1, Ec2 = 10.0, 12.0
    EJ1, EJ2 = 0.5,  0.7

    print("Building sparse Hamiltonian ...", flush=True)
    H_sp = ssh_squid_chain_sparse(
        N, fock_photon, fock_transmon,
        E_c, E_J, omega_c, gamma,
        Ec1, Ec2, EJ1, EJ2
    )
    print(f"  shape : {H_sp.shape}")
    print(f"  nnz   : {H_sp.nnz}")
    print(f"  dtype : {H_sp.dtype}")

    # # Check Hermiticity
    # diff = H_sp - H_sp.conj().T
    # print(f"  ||H - H†||_max = {np.abs(diff).max():.3e}")

    # Lowest few eigenvalues via ARPACK
    # from scipy.sparse.linalg import eigsh
    # vals, _ = eigsh(H_sp, k=6, which='SA')
    # print(f"  lowest eigenvalues: {np.sort(vals.real)}")

    # # --- QuTiP reference ---
    # # (import original functions from script.py if on path)
    # try:
    #     sys.path.insert(0, '/home/ruiz/Documents/thesis/notebooks/exploration')
    #     from script import ssh_squid_chain as qt_chain
    #     print("\nBuilding QuTiP Hamiltonian for comparison ...", flush=True)
    #     H_qt = qt_chain(
    #         N, fock_photon, fock_transmon,
    #         E_c, E_J, omega_c, gamma,
    #         Ec1, Ec2, EJ1, EJ2
    #     )
    #     H_qt_sp = sp.csr_matrix(H_qt.full())
    #     diff2 = H_sp - H_qt_sp
    #     print(f"  ||H_sparse - H_qutip||_max = {np.abs(diff2).max():.3e}")
    # except Exception as e:
    #     print(f"  (QuTiP comparison skipped: {e})")

Building sparse Hamiltonian ...
  shape : (4194304, 4194304)
  nnz   : 121634816
  dtype : complex128


In [6]:
# N = 9
# fock_photon = 3
# fock_transmon = 3

# E_c = 1.0
# E_J = 50.0
# omega_q = np.sqrt(8 * E_c * E_J) - E_c
# omega_c = omega_q/2.0 # fuera de resonancia

# gamma = 1.0e-3 # un valor bastante optimista para gamma
# Ec1 = E_c * 1.1
# Ec2 = E_c * 1.2

# E_J1 = 500.0
# E_J2 = -500.0

In [7]:
# H_sp = ssh_squid_chain_sparse(
#     N, fock_photon, fock_transmon,
#     E_c, E_J, omega_c, gamma,
#     Ec1, Ec2, E_J1, E_J2
# )

In [8]:
import numpy as np  
k_values = np.linspace(-np.pi, np.pi, 11, endpoint = False)

In [9]:
k_values

array([-3.14159265, -2.57039399, -1.99919533, -1.42799666, -0.856798  ,
       -0.28559933,  0.28559933,  0.856798  ,  1.42799666,  1.99919533,
        2.57039399])